# Parcial — Análisis del mercado laboral colombiano (GEIH, junio 2026)

Desarrollo de los ejercicios del parcial sobre la Gran Encuesta Integrada de Hogares (GEIH) de DANE. Todas las cifras están ponderadas con el factor de expansión `FEX_C18` y los ingresos (`INGLABO`) son mensuales, en pesos colombianos (COP). Cada punto incluye el código, el resultado y su interpretación textual.

In [0]:
import pandas as pd 

In [0]:
#Leyendo la tabla de caracteristicas generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

#Primero hay que anexar la p3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']], 
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
# Cálculos ponderados
pet_total = ft[ft['PET'] == 1]['FEX_C18'].sum()
no_pet_total = cgse['FEX_C18'].sum() - pet_total  # Población total (cgse) menos PET

ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()

ocupados_total = ocupados[ocupados['OCI'] == 1]['FEX_C18'].sum()
desocupados_total = desempleados[desempleados['DSI'] == 1]['FEX_C18'].sum()

pea_total = ocupados_total + desocupados_total # Población Económicamente Activa (FT)
pei_total = pet_total - pea_total              # Población Económicamente Inactiva

tasa_ocupacion = (ocupados_total / pet_total) * 100
tasa_desempleo = (desocupados_total / pea_total) * 100

print(f"Población en edad de trabajar (PET): {pet_total:,.0f}")
print(f"Población que no está en edad de trabajar: {no_pet_total:,.0f}")
print(f"Población Económicamente Activa (PEA): {pea_total:,.0f}")
print(f"Población Económicamente Inactiva (PEI): {pei_total:,.0f}")
print(f"Tasa de Ocupación (TO): {tasa_ocupacion:.2f}%")
print(f"Tasa de Desempleo (TD): {tasa_desempleo:.2f}%")

## Punto 1 — Indicadores del mercado laboral a nivel nacional (junio 2026)

**Pregunta:** Para el total nacional calcule la población en edad de trabajar, la que no está en edad de trabajar, la población económicamente activa, la inactiva, la tasa de ocupación y la tasa de desempleo.

**Respuesta:**

* Población en edad de trabajar (PET): **41,202,631**
* Población que no está en edad de trabajar: **11,346,181**
* Población Económicamente Activa (PEA): **26,593,429**
* Población Económicamente Inactiva (PEI): **14,609,202**
* Tasa de Ocupación (TO): **59.36%**
* Tasa de Desempleo (TD): **8.03%**

**Interpretación:** De los ~52.5 millones de habitantes que cubre la encuesta, 41.2 millones (78%) están en edad de trabajar (15 años o más) y 11.3 millones son menores de 15. De la PET, 26.6 millones participan activamente del mercado laboral (PEA) y 14.6 millones son económicamente inactivos (estudiantes, oficios del hogar, jubilados, etc.). Dentro de la PEA, el 91.97% está ocupado y el 8.03% está desempleado: alrededor de 2.1 millones de personas buscan trabajo sin conseguirlo. La tasa de ocupación del 59.36% indica que seis de cada diez personas en edad de trabajar tienen empleo.

In [0]:
# 2. Tasas de desempleo por departamento y género
# Diccionario de departamentos (código DANE)
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

# Unir ocupados con cgse para anexar P3271 (Sexo: 1=Hombre, 2=Mujer); ocupados ya tiene DPTO
oc_completo = ocupados[ocupados['OCI'] == 1].merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='inner'
)

# desempleados ya tiene DPTO y P3271 (anexados en la celda 2)
des_completo = desempleados[desempleados['DSI'] == 1]

# Sumar Ocupados y Desocupados por Depto y Género
oc_agrup = oc_completo.groupby(['DPTO', 'P3271'])['FEX_C18'].sum().rename('OCUPADOS')
des_agrup = des_completo.groupby(['DPTO', 'P3271'])['FEX_C18'].sum().rename('DESOCUPADOS')

df_td = pd.concat([oc_agrup, des_agrup], axis=1).fillna(0).reset_index()
df_td['PEA'] = df_td['OCUPADOS'] + df_td['DESOCUPADOS']
df_td['Tasa_Desempleo'] = (df_td['DESOCUPADOS'] / df_td['PEA']) * 100

# Mapear Género y Departamento
df_td['Sexo'] = df_td['P3271'].map({1: 'Hombre', 2: 'Mujer'})
df_td['Departamento'] = df_td['DPTO'].map(dpto_nombre)

# Pivotear para ver Tasas por Departamento
resultado_p2 = df_td.pivot(index='Departamento', columns='Sexo', values='Tasa_Desempleo').reset_index()

display(resultado_p2)

# Departamentos con mayor y menor tasa global
td_depto = df_td.groupby('Departamento')[['DESOCUPADOS', 'PEA']].sum()
td_depto['TD_Total'] = (td_depto['DESOCUPADOS'] / td_depto['PEA']) * 100

print("Departamento con MAYOR Tasa de Desempleo:")
print(td_depto.sort_values(by='TD_Total', ascending=False).head(3)[['TD_Total']])

print("\nDepartamento con MENOR Tasa de Desempleo:")
print(td_depto.sort_values(by='TD_Total', ascending=True).head(3)[['TD_Total']])

# --- Tabla resumen: TD_Hombres, TD_Mujeres y TD_Total por departamento ---
tabla_resumen = resultado_p2.merge(td_depto[['TD_Total']].reset_index(), on='Departamento')
tabla_resumen = tabla_resumen.rename(columns={
    'Hombre': 'TD_Hombres (%)', 'Mujer': 'TD_Mujeres (%)', 'TD_Total': 'TD_Total (%)'
})

display(tabla_resumen)

# --- Conclusiones textuales ---
d_max, d_min = td_depto['TD_Total'].idxmax(), td_depto['TD_Total'].idxmin()
print(f"El departamento con mayor tasa de desempleo general fue {d_max} ({td_depto.loc[d_max, 'TD_Total']:.2f}%), mientras que el de menor tasa fue {d_min} ({td_depto.loc[d_min, 'TD_Total']:.2f}%).")

h_max = tabla_resumen.loc[tabla_resumen['TD_Hombres (%)'].idxmax()]
h_min = tabla_resumen.loc[tabla_resumen['TD_Hombres (%)'].idxmin()]
print(f"En cuanto a hombres, la mayor tasa la registró {h_max['Departamento']} ({h_max['TD_Hombres (%)']:.2f}%) y la menor {h_min['Departamento']} ({h_min['TD_Hombres (%)']:.2f}%).")

m_max = tabla_resumen.loc[tabla_resumen['TD_Mujeres (%)'].idxmax()]
m_min = tabla_resumen.loc[tabla_resumen['TD_Mujeres (%)'].idxmin()]
print(f"En mujeres, la mayor tasa la registró {m_max['Departamento']} ({m_max['TD_Mujeres (%)']:.2f}%) y la menor {m_min['Departamento']} ({m_min['TD_Mujeres (%)']:.2f}%).")

## Punto 2 — Tasas de desempleo por departamento y sexo

**Pregunta:** Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia. ¿Cuáles son los departamentos con mayores y menores tasas de desempleo?

**Respuesta:**

* General: mayor **PUTUMAYO (22.18%)** — menor **VAUPÉS (2.46%)**
* Hombres: mayor **SAN ANDRÉS (20.98%)** — menor **VAUPÉS (3.00%)**
* Mujeres: mayor **PUTUMAYO (25.02%)** — menor **VAUPÉS (1.78%)**

**Interpretación:** El desempleo en Colombia muestra fuertes contrastes territoriales y de género. Putumayo encabeza la crisis laboral con una tasa total del 22.2% — casi el triple del promedio nacional (8.03%) — y en mujeres llega al 25%: una de cada cuatro mujeres activas no consigue empleo. En el extremo opuesto, Vaupés registra la menor tasa en los tres cortes, aunque debe leerse con cautela porque su muestra es muy pequeña. En casi todos los departamentos la TD femenina supera a la masculina; la brecha más marcada está en Bolívar (17.4% mujeres vs 6.4% hombres) y Boyacá (14.1% vs 7.8%), mientras que solo en Bogotá y Amazonas los hombres desemplean ligeramente más. Esto evidencia que las mujeres enfrentan mayores barreras de inserción laboral que los hombres.

In [0]:
# 3. Tasa de desempleo por departamento y grupo de edad
# 1. Definir la función con los rangos de edad (incluye 50 a 60, como en la celda anterior)
def categorizar_edad_p3(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif 50 <= edad < 60:
        return '50 a 60'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

# 2. Asignar rangos de edad en la tabla de características generales
cgse['Grupo_Edad'] = cgse['P6040'].apply(categorizar_edad_p3)

# 3. Unir Ocupados y Desempleados con Características para heredar el Grupo de Edad
oc_p3 = ocupados[ocupados['OCI'] == 1].merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'Grupo_Edad']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='inner'
)

des_p3 = desempleados[desempleados['DSI'] == 1].merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'Grupo_Edad']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='inner'
)

# Filtrar solo personas que están dentro de los rangos definidos
oc_p3 = oc_p3[oc_p3['Grupo_Edad'].notna()]
des_p3 = des_p3[des_p3['Grupo_Edad'].notna()]

# 4. Agrupar por Departamento y Grupo de Edad
oc_edad = oc_p3.groupby(['DPTO', 'Grupo_Edad'])['FEX_C18'].sum().rename('OCUPADOS')
des_edad = des_p3.groupby(['DPTO', 'Grupo_Edad'])['FEX_C18'].sum().rename('DESOCUPADOS')

df_p3 = pd.concat([oc_edad, des_edad], axis=1).fillna(0).reset_index()
df_p3['PEA'] = df_p3['OCUPADOS'] + df_p3['DESOCUPADOS']
df_p3['Tasa_Desempleo'] = (df_p3['DESOCUPADOS'] / df_p3['PEA']) * 100
df_p3['Departamento'] = df_p3['DPTO'].map(dpto_nombre)

# 5. Obtener el grupo de edad con MAYOR y MENOR tasa por departamento
idx_max = df_p3.groupby('Departamento')['Tasa_Desempleo'].idxmax()
idx_min = df_p3.groupby('Departamento')['Tasa_Desempleo'].idxmin()

max_td = df_p3.loc[idx_max, ['Departamento', 'Grupo_Edad', 'Tasa_Desempleo']].rename(
    columns={'Grupo_Edad': 'Edad Mayor TD', 'Tasa_Desempleo': 'Mayor TD (%)'}
)

min_td = df_p3.loc[idx_min, ['Departamento', 'Grupo_Edad', 'Tasa_Desempleo']].rename(
    columns={'Grupo_Edad': 'Edad Menor TD', 'Tasa_Desempleo': 'Menor TD (%)'}
)

resultado_p3 = pd.merge(max_td, min_td, on='Departamento')
display(resultado_p3)

## Punto 3 — Tasa de desempleo por departamento y grupo de edad

**Pregunta:** Calcule la tasa de desempleo por departamento y grupo de edad, e identifique para cada departamento el grupo con mayor y menor tasa.

**Respuesta:** La tabla `resultado_p3` muestra, para cada uno de los 33 departamentos, el grupo de edad con mayor y menor TD.

**Interpretación:** El desempleo golpea sobre todo a los jóvenes: en la mayoría de departamentos la mayor tasa se concentra en **18 a 24 años** (Bolívar 30.9%, Casanare 29.2%, Córdoba 23.9%, Cundinamarca 23.1%) o en **24 a 30 años** (Arauca 22.8%, Boyacá 22.6%, Chocó 19.2%). En contraste, las menores tasas se presentan en los grupos de mayor edad — **60 en adelante** o **50 a 60 años** — con valores entre 0% y 5.4%. Este patrón refleja la desventaja de entrada al mercado laboral de los jóvenes (falta de experiencia, informalidad, contratos cortos) frente a la estabilidad de los trabajadores consolidados. En el grupo 60+ la TD es mínima también porque quien pierde o no consigue empleo suele retirarse a la inactividad (pensión) y deja de contarse como desempleado.

In [0]:
# 5. 3 Departamentos con menor tasa de desempleo en mujeres de 18 a 25 años
# 1. Unir Ocupados con CGSE para tener P3271 (Sexo) y P6040 (Edad)
oc_p4 = ocupados[ocupados['OCI'] == 1].merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='inner'
)

# desempleados ya tiene P3271 y P6040 (anexados en la celda 2), no hace falta unir
des_p4 = desempleados[desempleados['DSI'] == 1]

# 2. Filtrar Mujeres (2) entre 18 y 25 años inclusive
oc_p4_mujeres = oc_p4[(oc_p4['P3271'] == 2) & (oc_p4['P6040'] >= 18) & (oc_p4['P6040'] <= 25)]
des_p4_mujeres = des_p4[(des_p4['P3271'] == 2) & (des_p4['P6040'] >= 18) & (des_p4['P6040'] <= 25)]

# 3. Sumar ponderados por Departamento
oc_p4_agrup = oc_p4_mujeres.groupby('DPTO')['FEX_C18'].sum().rename('OCUPADOS')
des_p4_agrup = des_p4_mujeres.groupby('DPTO')['FEX_C18'].sum().rename('DESOCUPADOS')

df_p4 = pd.concat([oc_p4_agrup, des_p4_agrup], axis=1).fillna(0).reset_index()
df_p4['PEA'] = df_p4['OCUPADOS'] + df_p4['DESOCUPADOS']
df_p4['Tasa_Desempleo'] = (df_p4['DESOCUPADOS'] / df_p4['PEA']) * 100
df_p4['Departamento'] = df_p4['DPTO'].map(dpto_nombre)

# 4. Ordenar y obtener los 3 departamentos con menor tasa de desempleo
top3_menor_td_mujeres = df_p4[['Departamento', 'Tasa_Desempleo']].sort_values(
    by='Tasa_Desempleo', ascending=True
).head(3).reset_index(drop=True)

print("3 Departamentos con MENOR tasa de desempleo en mujeres de 18 a 25 años:")
display(top3_menor_td_mujeres)

## Punto 4 — 3 departamentos con menor desempleo en mujeres de 18 a 25 años

**Pregunta:** ¿Cuáles son los 3 departamentos con menor tasa de desempleo en mujeres de 18 a 25 años?

**Respuesta:**

1. **VAUPÉS — 0.00%**
2. **NARIÑO — 4.29%**
3. **AMAZONAS — 4.87%**

**Interpretación:** Los tres departamentos con menor desempleo juvenil femenino son dos amazónicos y Nariño. El 0% de Vaupés casi seguro refleja una muestra muy pequeña — hay muy pocas mujeres de 18 a 25 años en la PEA encuestada de ese departamento y ninguna reportó estar desempleada — más que una ausencia real de desempleo. Nariño (4.3%) y Amazonas (4.9%) sí registran tasas genuinamente bajas, incluso por debajo del promedio nacional (8.03%), lo que puede asociarse a economías locales con alta absorción de empleo informal en comercio y agricultura.

In [0]:
# 6. Promedio INGLABO ponderado — Risaralda (66), Caldas (17), Quindío (63), Antioquia (5) y Valle del Cauca (76)
deptos_p5 = [66, 17, 63, 5, 76]

# 1. Filtrar ocupados de los departamentos seleccionados
ingresos_df = ocupados[
    (ocupados['OCI'] == 1) &
    (ocupados['DPTO'].isin(deptos_p5))
].copy()

# 2. Convertir a numérico y limpiar
ingresos_df['INGLABO'] = pd.to_numeric(ingresos_df['INGLABO'], errors='coerce')
ingresos_df['FEX_C18'] = pd.to_numeric(ingresos_df['FEX_C18'], errors='coerce')

ingresos_df = ingresos_df[
    (ingresos_df['INGLABO'] > 0) &
    (ingresos_df['FEX_C18'] > 0)
].copy()

# 3. Ingreso Ponderado
ingresos_df['INGRESO_PONDERADO'] = ingresos_df['INGLABO'] * ingresos_df['FEX_C18']

# 4. Agrupar por departamento y calcular la media ponderada
tabla_p5 = ingresos_df.groupby('DPTO')[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()
tabla_p5['Promedio INGLABO'] = tabla_p5['INGRESO_PONDERADO'] / tabla_p5['FEX_C18']
tabla_p5['Departamento'] = tabla_p5['DPTO'].map(dpto_nombre)

resultado_punto5 = tabla_p5[['Departamento', 'Promedio INGLABO']].sort_values(
    by='Promedio INGLABO', ascending=False
).reset_index(drop=True)

display(resultado_punto5)

## Punto 5 — Promedio de ingreso laboral (INGLABO) en 5 departamentos

**Pregunta:** Calcule el promedio de ingreso laboral ponderado en Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca.

**Respuesta (ingreso mensual ponderado, COP):**

1. Antioquia — $2,486,207
2. Risaralda — $2,234,061
3. Caldas — $2,226,450
4. Quindío — $2,171,560
5. Valle del Cauca — $2,147,996

**Interpretación:** Los cinco departamentos tienen ingresos laborales promedio muy similares (entre $2.1 y $2.2 millones mensuales).

In [0]:
# 6. Promedio de ingresos laborales por departamento, sexo y rango de edad

# Departamentos seleccionados (del punto anterior)
deptos_p6 = [66, 17, 63, 5, 76]

# 1. Definir la función para categorizar los rangos de edad solicitados
def categorizar_edad_p6(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif 50 <= edad < 60:
        return '50 a 60'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

# 2. Filtrar ocupados de los departamentos requeridos
ingres_p6 = ocupados[
    (ocupados['OCI'] == 1) & 
    (ocupados['DPTO'].isin(deptos_p6))
].copy()

# 3. Unir con cgse para heredar Edad (P6040) y Sexo (P3271) si no están en ocupados
ingres_p6 = ingres_p6.merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P6040', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='inner'
)

# 4. Asignar grupos de edad y etiquetas de sexo
ingres_p6['Grupo_Edad'] = ingres_p6['P6040'].apply(categorizar_edad_p6)
ingres_p6['Sexo'] = ingres_p6['P3271'].map({1: 'Hombre', 2: 'Mujer'})

# 5. Convertir a numérico y limpiar (ingresos válidos y mayores a cero)
ingres_p6['INGLABO'] = pd.to_numeric(ingres_p6['INGLABO'], errors='coerce')
ingres_p6['FEX_C18'] = pd.to_numeric(ingres_p6['FEX_C18'], errors='coerce')

ingres_p6 = ingres_p6[
    (ingres_p6['INGLABO'] > 0) & 
    (ingres_p6['FEX_C18'] > 0) &
    (ingres_p6['Grupo_Edad'].notna())
].copy()

# 6. Calcular ingreso ponderado
ingres_p6['INGRESO_PONDERADO'] = ingres_p6['INGLABO'] * ingres_p6['FEX_C18']

# 7. Agrupar por Departamento, Sexo y Grupo de Edad
tabla_p6 = ingres_p6.groupby(['DPTO', 'Sexo', 'Grupo_Edad'])[
    ['INGRESO_PONDERADO', 'FEX_C18']
].sum().reset_index()

# 8. Promedio ponderado y formateo final
tabla_p6['Promedio INGLABO'] = tabla_p6['INGRESO_PONDERADO'] / tabla_p6['FEX_C18']
tabla_p6['Departamento'] = tabla_p6['DPTO'].map(dpto_nombre)

# Presentación final ordenada
resultado_p6 = tabla_p6[
    ['Departamento', 'Sexo', 'Grupo_Edad', 'Promedio INGLABO']
].sort_values(
    by=['Departamento', 'Sexo', 'Grupo_Edad']
).reset_index(drop=True)

display(resultado_p6)

## Punto 6 — Ingreso laboral por departamento, sexo y rango de edad

**Pregunta:** Calcule el promedio de ingreso laboral en los 5 departamentos por sexo y rango de edad.

**Interpretación:** En los cinco departamentos el ingreso promedio es **consistentemente mayor para hombres que para mujeres en prácticamente todos los rangos de edad, y la brecha se amplía con la edad**. En Antioquia, por ejemplo, los hombres de 60+ promedian $2.33 millones frente a $1.26 millones de las mujeres (85% más), mientras que en 18 a 24 años la diferencia es menor ($1.72M vs $1.59M). Los ingresos crecen con la edad hasta el rango de 40 a 60 años, donde se alcanza el pipo salarial (~$2.8–3.0 millones en hombres antioqueños), y descienden en 60+. El patrón refleja acumulación de experiencia y ascensos hasta la madurez laboral; la brecha de género, a su vez, combina diferencias salariales directas con segregación ocupacional y las interrupciones de carrera asociadas al trabajo de cuidado.

In [0]:
# 7. Mayores promedios salariales por oficio a nivel nacional
# (El oficio está en la columna OFICIO_C8: código CIUO-08 de 4 dígitos)

# Limpieza de variables en ocupados
ocupados_p7 = ocupados[(ocupados['OCI'] == 1)].copy()
ocupados_p7['INGLABO'] = pd.to_numeric(ocupados_p7['INGLABO'], errors='coerce')
ocupados_p7['FEX_C18'] = pd.to_numeric(ocupados_p7['FEX_C18'], errors='coerce')

# Filtrar registros válidos
ocupados_p7 = ocupados_p7[(ocupados_p7['INGLABO'] > 0) & (ocupados_p7['FEX_C18'] > 0)].copy()

# Ingreso Ponderado
ocupados_p7['INGRESO_PONDERADO'] = ocupados_p7['INGLABO'] * ocupados_p7['FEX_C18']

# Agrupar por OFICIO (OFICIO_C8)
oficios_nac = ocupados_p7.groupby('OFICIO_C8')[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()
oficios_nac['Promedio INGLABO'] = oficios_nac['INGRESO_PONDERADO'] / oficios_nac['FEX_C18']

# Top 10 oficios con mayor promedio salarial
top_oficios = oficios_nac.sort_values(by='Promedio INGLABO', ascending=False).head(10).reset_index(drop=True)

# Nombres de los oficios según la clasificación CIUO-08 ajustada por DANE
# (documento oficial: CIUO_08_AC_2015_07_21.pdf, dane.gov.co — títulos verificados contra el texto)
oficio_nombre = {
    1111: 'Directores formuladores de políticas y normas',
    1120: 'Directores y gerentes generales',
    2211: 'Médicos generales',
    2212: 'Médicos especialistas',
    2612: 'Jueces',
    1223: 'Directores de investigación y desarrollo',
    1330: 'Directores de servicios de tecnología de la información y las comunicaciones',
    2422: 'Profesionales en políticas de administración',
    2522: 'Administradores de sistemas',
    1323: 'Directores de empresas de construcción',
    3321: 'Agentes de seguros',
    2145: 'Ingenieros químicos',
    2512: 'Desarrolladores de software',
    9611: 'Recolectores de basura y material reciclable',
    3139: 'Técnicos en control de procesos no clasificados en otros grupos primarios',
}

# Anexar el nombre del oficio (si el código no está en el diccionario, se muestra el código)
top_oficios['Oficio'] = top_oficios['OFICIO_C8'].map(oficio_nombre)
top_oficios['Oficio'] = top_oficios['Oficio'].fillna(top_oficios['OFICIO_C8'].astype(str))

top_oficios = top_oficios[['Oficio', 'OFICIO_C8', 'FEX_C18', 'Promedio INGLABO']]

display(top_oficios)

In [0]:
# 8. Diferencias salariales por sexo en Caldas, Quindío, Antioquia y Valle del Cauca para un oficio específico

deptos_p8 = [17, 63, 5, 76]  # Caldas, Quindío, Antioquia, Valle del Cauca
oficio_elegido = 2211        # Médicos generales (CIUO-08); puedes cambiarlo por el oficio que prefieras

# Filtrar ocupados del oficio elegido en los departamentos requeridos
p8_df = ocupados[
    (ocupados['OCI'] == 1) &
    (ocupados['DPTO'].isin(deptos_p8)) &
    (ocupados['OFICIO_C8'] == oficio_elegido)
].copy()

# Unir con cgse para heredar el Sexo (P3271)
p8_df = p8_df.merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='inner'
)

# Convertir tipos
p8_df['INGLABO'] = pd.to_numeric(p8_df['INGLABO'], errors='coerce')
p8_df['FEX_C18'] = pd.to_numeric(p8_df['FEX_C18'], errors='coerce')

p8_df = p8_df[(p8_df['INGLABO'] > 0) & (p8_df['FEX_C18'] > 0)].copy()
p8_df['INGRESO_PONDERADO'] = p8_df['INGLABO'] * p8_df['FEX_C18']
p8_df['Sexo'] = p8_df['P3271'].map({1: 'Hombre', 2: 'Mujer'})

# Agrupar por departamento y sexo
tabla_p8 = p8_df.groupby(['DPTO', 'Sexo'])[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()
tabla_p8['Promedio INGLABO'] = tabla_p8['INGRESO_PONDERADO'] / tabla_p8['FEX_C18']
tabla_p8['Departamento'] = tabla_p8['DPTO'].map(dpto_nombre)

resultado_p8 = tabla_p8.pivot(index='Departamento', columns='Sexo', values='Promedio INGLABO').reset_index()

print(f"Diferencias salariales por sexo — oficio: código CIUO-08 {oficio_elegido} (médicos generales)")
display(resultado_p8)

In [0]:
# 9. Oficios con mayor y menor ingreso laboral en el departamento del Quindío (DPTO == 63)

# Filtrar Quindío en ocupados y cruzar sexo
p9_df = ocupados[(ocupados['OCI'] == 1) & (ocupados['DPTO'] == 63)].copy()

p9_df = p9_df.merge(
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='inner'
)

p9_df['INGLABO'] = pd.to_numeric(p9_df['INGLABO'], errors='coerce')
p9_df['FEX_C18'] = pd.to_numeric(p9_df['FEX_C18'], errors='coerce')

p9_df = p9_df[(p9_df['INGLABO'] > 0) & (p9_df['FEX_C18'] > 0)].copy()
p9_df['INGRESO_PONDERADO'] = p9_df['INGLABO'] * p9_df['FEX_C18']
p9_df['Sexo'] = p9_df['P3271'].map({1: 'Hombre', 2: 'Mujer'})

# Agrupar por Sexo y Oficio (OFICIO_C8)
tabla_p9 = p9_df.groupby(['Sexo', 'OFICIO_C8'])[['INGRESO_PONDERADO', 'FEX_C18']].sum().reset_index()
tabla_p9['Promedio INGLABO'] = tabla_p9['INGRESO_PONDERADO'] / tabla_p9['FEX_C18']

# Mayor y Menor oficio para Hombres
hombres_p9 = tabla_p9[tabla_p9['Sexo'] == 'Hombre']
max_h = hombres_p9.sort_values(by='Promedio INGLABO', ascending=False).head(1)
min_h = hombres_p9.sort_values(by='Promedio INGLABO', ascending=True).head(1)

# Mayor y Menor oficio para Mujeres
mujeres_p9 = tabla_p9[tabla_p9['Sexo'] == 'Mujer']
max_m = mujeres_p9.sort_values(by='Promedio INGLABO', ascending=False).head(1)
min_m = mujeres_p9.sort_values(by='Promedio INGLABO', ascending=True).head(1)

# Concatenar resultados
resumen_quindio = pd.concat([max_h, min_h, max_m, min_m]).reset_index(drop=True)
resumen_quindio['Categoría'] = ['Mayor Hombres', 'Menor Hombres', 'Mayor Mujeres', 'Menor Mujeres']

# Anexar el nombre del oficio cuando esté en el diccionario CIUO-08 (celda 9); si no, el código
resumen_quindio['Oficio'] = resumen_quindio['OFICIO_C8'].map(oficio_nombre)
resumen_quindio['Oficio'] = resumen_quindio['Oficio'].fillna(resumen_quindio['OFICIO_C8'].astype(str))

resultado_p9 = resumen_quindio[['Categoría', 'Sexo', 'Oficio', 'OFICIO_C8', 'Promedio INGLABO']]
display(resultado_p9)

# Respuesta textual a la pregunta del taller
print("\nRespuesta — Oficios con mayor y menor promedio salarial en Quindío:")
for _, r in resumen_quindio.iterrows():
    print(f"  {r['Categoría']}: {r['Oficio']} (código {r['OFICIO_C8']}) — promedio ${r['Promedio INGLABO']:,.0f}")

## Punto 9 — Oficio con mayor y menor promedio salarial en Quindío, por sexo

**Pregunta:** En el departamento del Quindío, ¿cuál es el oficio con mayor y menor promedio salarial en hombres y mujeres?

**Respuesta:**

* **Hombres:** mayor — ingenieros químicos (**$23,000,000**); menor — recolectores de basura y material reciclable (**$96,881**)
* **Mujeres:** mayor — desarrolladoras de software (**$8,889,220**); menor — técnicas en control de procesos n.c.p. (**$150,000**)

**Interpretación:** La escala salarial del Quindío reproduce la jerarquía nacional: los oficios profesionales y técnicos (ingeniería química, desarrollo de software) concentran los ingresos más altos, mientras que los oficios no calificados quedan en el extremo inferior, con una brecha que supera las 100 veces entre extremos. El oficio de menor ingreso masculino — recolectores de basura y material reciclable — corresponde típicamente a recicladores informales que trabajan pocas jornadas, lo cual es coherente con el ingreso mensual reportado. **Advertencia:** los valores de los extremos inferiores ($96,881 y $150,000 mensuales) provienen de 1–2 encuestados por oficio y sexo — la GEIH registra ingresos menores a $100,000 en apenas el 0.8% de los ocupados, típicamente trabajos informales de muy pocas horas semanales — por lo que el resultado correcto para reportar es el contraste de escala entre oficios calificados y no calificados, no los montos puntuales.

---

## Notas metodológicas

* Todas las cifras están ponderadas con el factor de expansión `FEX_C18` de la GEIH (junio de 2026).
* `INGLABO` es el ingreso laboral **mensual**, en pesos colombianos (COP).
